# BioBERT Relation Extraction - Kaggle Execution Controller

This notebook serves as the execution controller for fine-tuning BioBERT on the n2c2 2010 relation extraction dataset.
The training logic is entirely modularized in the `src/` directory. This notebook orchestrates the environment setup, configuration writing, launching the training script with `accelerate`, executing evaluation, and backing up outputs.

## 1. Setup Environment and Clone Repo

In [ ]:
# Clone the repository if not already in the repository directory
# !git clone <repo_url> vi_medical_re
# %cd vi_medical_re

# Install dependencies
!pip install -r requirements.txt

# Create required directory structure
!mkdir -p outputs logs checkpoints configs

# Record reproducibility details
!git rev-parse HEAD > outputs/git_commit.txt 2>/dev/null || echo "Not a git repo" > outputs/git_commit.txt
!pip freeze > outputs/environment.txt

## 2. Configuration Management
Modify the configuration in this cell. It will write directly to `configs/train.yaml` and `configs/eval.yaml`.

In [ ]:
import yaml
import os

TRAIN_CONFIG = {
    "experiment_name": "2026_07_14_biobert_n2c2",
    "model_name_or_path": "dmis-lab/biobert-base-cased-v1.2",
    "train_file": "n2c2 2010/n2c2_2010_train_en.jsonl",
    "dev_file": "n2c2 2010/n2c2_2010_test_en.jsonl",
    "epochs": 5,
    "batch_size": 8,
    "learning_rate": 2e-5,
    "max_seq_length": 128,
    "gradient_accumulation_steps": 4,
    "max_grad_norm": 1.0,
    "mixed_precision": "fp16",
    "save_steps": 500,
    "eval_steps": 500,
    "seed": 42,
    "output_dir": "outputs/2026_07_14_biobert_n2c2",
    "checkpoint_dir": "checkpoints",
    "log_dir": "logs"
}

EVAL_CONFIG = {
    "experiment_name": "2026_07_14_biobert_n2c2",
    "model_name_or_path": "dmis-lab/biobert-base-cased-v1.2",
    "test_file": "n2c2 2010/n2c2_2010_test_en.jsonl",
    "batch_size": 16,
    "max_seq_length": 128,
    "checkpoint_path": "checkpoints/latest",
    "output_dir": "outputs/2026_07_14_biobert_n2c2",
    "log_dir": "logs"
}

# Write YAML files
os.makedirs("configs", exist_ok=True)
with open("configs/train.yaml", "w") as f:
    yaml.dump(TRAIN_CONFIG, f, default_flow_style=False)
with open("configs/eval.yaml", "w") as f:
    yaml.dump(EVAL_CONFIG, f, default_flow_style=False)

print("YAML configs successfully updated.")

## 3. Check for Existing Checkpoints (Resume Logic)

In [ ]:
import os

resume_flag = ""
if os.path.exists("checkpoints/latest") and os.path.exists("checkpoints/latest/trainer_state.json"):
    print("=== Found existing checkpoint checkpoints/latest. Resuming is enabled. ===")
    resume_flag = "--resume_from_checkpoint"
else:
    print("=== No checkpoints found. Model will train from scratch. ===")

## 4. Run Multi-GPU Training via Accelerate

In [ ]:
!accelerate launch \
  --config_file accelerate_config.yaml \
  src/train.py \
  --config configs/train.yaml \
  {resume_flag}

## 5. View Training Logs

In [ ]:
if os.path.exists("logs/train/train.log"):
    print("=== Printing last 40 lines of train.log ===")
    with open("logs/train/train.log", "r") as f:
        lines = f.readlines()
        for line in lines[-40:]:
            print(line.strip())

## 6. Run Evaluation on Test Set

In [ ]:
!python src/evaluate.py --config configs/eval.yaml

## 7. Display Evaluation Metrics

In [ ]:
import json
results_path = "outputs/2026_07_14_biobert_n2c2/results.json"
if os.path.exists(results_path):
    with open(results_path, "r") as f:
        results = json.load(f)
        print(json.dumps(results, indent=4))

## 8. Backup Output Artifacts

In [ ]:
# Archive outputs, checkpoints, and logs for download
!tar -czf experiment_backup.tar.gz outputs logs checkpoints
print("Packaged backups into experiment_backup.tar.gz")